# STAIR5-v8 — WMSG-CSE 256D | Kaggle training

Thí nghiệm **capacity-only**: MI/FSC/BSC/BPR/NLGCL, graph WMSG-CSE, optimizer và evaluator kế thừa V8. Không thêm consensus, SAP, relation gate hay UCR. WMSG dùng power-scaled cosine trên cached top-k, không dùng trọng số inverse-degree được mô tả nhầm trong notebook cũ.

**Chạy tuần tự các cell setup → pre-flight → runner**, rồi chọn cell Sports, Baby hoặc Electronics. Mỗi tập có thể chạy độc lập; không bị khóa bởi một danh sách “Not selected”. Mặc định **500 epochs**, validation mỗi 5 epochs, chọn checkpoint bằng validation NDCG@20. `MODE="smoke"` chỉ chạy 3 epochs và lưu namespace riêng.

Bật Internet và GPU trên Kaggle, gắn dữ liệu `rainyle/stair-datasets-mmrec`. 256D tốn nhiều thời gian/bộ nhớ hơn 64D; không cam kết dưới 800 MiB hay hoàn thành Electronics trong một session. Checkpoint và archive cần được lưu ra ngoài session để resume. Chưa có kết quả 256D; các số V8-64 chỉ là tham chiếu lịch sử.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess, uuid

REPOSITORY_URL = "https://github.com/ThanhChuong12/STAIR-Enhanced.git"
SOURCE_REF = "main"  # For reproduction/resume, replace with the recorded commit SHA.
WORK = Path("/kaggle/working")
REPO = WORK / "STAIR-Enhanced-v8-256D"
DATA_ROOT = WORK / "stair5_v8_256d_data"
OUTPUT_ROOT = WORK / "stair5_v8_256d_runs"
MODE = "full"  # "full" or "smoke"; never resume a smoke checkpoint into a full experiment.
SEEDS = (1,)  # Use (1, 2, 3) for the final multi-seed study; runs are sequential.
ACTIVE_DATASETS = ("sports", "baby", "electronics")  # Used only by the optional run_all().
RESUME_FROM = {}  # Example: {("electronics", 1): "/kaggle/input/backup/.../training_checkpoint.pt"}
SUPPORT_FILES = {}  # Optional frozen V8 candidates: {"sports": "/path/text.pt,/path/visual.pt"}
ALLOW_KAGGLEHUB_DOWNLOAD = False  # Prefer attaching the dataset as Kaggle Input.
DEVICE = "0"
NVML_DEVICE_INDEX = 0  # Physical GPU index; adjust if CUDA_VISIBLE_DEVICES changes mapping.
INSTALL_DEPENDENCIES = True
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
assert MODE in ("full", "smoke")
assert SEEDS and len(set(SEEDS)) == len(SEEDS)
assert all(isinstance(s, int) and not isinstance(s, bool) and s >= 0 for s in SEEDS)
EPOCHS, EVAL_FREQ = (500, 5) if MODE == "full" else (3, 1)
SESSION = OUTPUT_ROOT / SESSION_ID
SESSION.mkdir(parents=True, exist_ok=False)
print({"session": str(SESSION), "mode": MODE, "epochs": EPOCHS, "seeds": SEEDS})


## 1. Source synchronization and runtime

Dùng checkout riêng cho 256D. Fetch/checkout thất bại sẽ dừng; không hard-reset hoặc xóa mã nguồn/artifact cũ. Khi resume, pin `SOURCE_REF` về SHA đã chạy. Giữ wheel CUDA PyTorch do Kaggle cung cấp; cài FreeRec 0.9.7 không kéo theo một bản PyTorch khác. Compatibility shim của repository xử lý TorchData trước khi import FreeRec.


In [ ]:
import importlib.metadata as metadata

WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPO)], check=True)
if not (REPO / ".git").exists():
    raise RuntimeError(f"Expected a dedicated Git checkout at {REPO}")
dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
if dirty:
    raise RuntimeError("Checkout has tracked changes. Preserve them before changing SOURCE_REF.")
actual_origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
if actual_origin != REPOSITORY_URL:
    raise RuntimeError(f"Unexpected repository origin: {actual_origin}")
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", SOURCE_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
SOURCE_SHA = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
required = ["main_stair5_v8_256d.py", "stair5_v8_256d_config.py",
            "stair5_v8_256d_notebook_utils.py", "models/stair5_v8_256d.py",
            "tests/test_stair5_v8_256d_pipeline.py"]
for relative in required:
    if not (REPO / relative).is_file():
        raise FileNotFoundError(f"{relative} missing at {SOURCE_SHA}; fetch the 256D commit.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
SUB_ENV = os.environ.copy()
SUB_ENV["PYTHONPATH"] = str(REPO) + os.pathsep + SUB_ENV.get("PYTHONPATH", "")
SUB_ENV["PYTHONUTF8"] = "1"
SUB_ENV["PYTHONUNBUFFERED"] = "1"
SUB_ENV.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
SUB_ENV["PYTHONHASHSEED"] = "0"  # Fixed interpreter hash seed; training RNG uses --seed.
torch_before = metadata.version("torch")
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "freerec==0.9.7"], check=True)
    from packaging.requirements import Requirement
    # Respect installed CUDA Torch even when a transitive dependency requires Torch.
    constraints = SESSION / "torch-constraint.txt"
    constraints.write_text("torch==" + torch_before + "\n", encoding="utf-8")
    packages = ["PyYAML>=6,<7", "numpy>=1.26,<3", "scipy>=1.11,<2",
                "torch-geometric>=2.5,<3", "matplotlib>=3.7,<4", "nvidia-ml-py", "kagglehub"]
    for spec in metadata.requires("freerec") or []:
        requirement = Requirement(spec)
        if requirement.name.lower().replace("_", "-") in ("torch", "torchdata"):
            continue
        if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
            packages.append(str(requirement))
    subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), *packages], check=True)
if metadata.version("torch") != torch_before:
    raise RuntimeError("The CUDA PyTorch distribution changed; recreate the Kaggle environment.")

verify = """
import json, sys, torch
import models.freerec_compat
import freerec
from models.stair5_v8_256d import STAIR5_v8_256D_Model
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before training")
print(json.dumps({"python": sys.version.split()[0], "torch": str(torch.__version__),
 "freerec": freerec.__version__, "gpu": torch.cuda.get_device_name(0),
 "device_total_mib": torch.cuda.get_device_properties(0).total_memory / 2**20}))
"""
verified = subprocess.run([sys.executable, "-c", verify], cwd=REPO, env=SUB_ENV,
                          check=False, capture_output=True, text=True)
print(verified.stdout)
if verified.stderr:
    print(verified.stderr)
verified.check_returncode()
(SESSION / "environment.txt").write_text(verified.stdout + verified.stderr, encoding="utf-8")
(SESSION / "packages.json").write_text(json.dumps(
    sorted((d.metadata["Name"], d.version) for d in metadata.distributions()), indent=2), encoding="utf-8")
(SESSION / "source.json").write_text(json.dumps(
    {"repository": REPOSITORY_URL, "requested_ref": SOURCE_REF, "commit": SOURCE_SHA}, indent=2), encoding="utf-8")
print("Source:", SOURCE_SHA)


## 2. Dataset discovery

Mỗi source phải có đủ `train.txt`, `valid.txt`, `test.txt`, `textual_modality.pkl`, `visual_modality.pkl`. Tạo thư mục dataset có thể ghi cache và symlink từng split/feature vào root mới, không đổi split, convert feature hay ánh xạ ID. Nếu thiếu một tập, các tập khác vẫn có thể chạy; gọi tập thiếu sẽ báo lỗi rõ ràng.

KaggleHub chỉ dùng cơ chế xác thực chuẩn của môi trường; notebook không chứa hoặc tự ghi API token.


In [ ]:
from stair5_v8_256d_notebook_utils import DATASETS, CONFIGS, REQUIRED_DATA, link_processed_dataset
SEARCH_ROOTS = [Path("/kaggle/input"), REPO / "data"]
if ALLOW_KAGGLEHUB_DOWNLOAD:
    import kagglehub
    SEARCH_ROOTS.append(Path(kagglehub.dataset_download("rainyle/stair-datasets-mmrec")))
processed = DATA_ROOT / "Processed"
processed.mkdir(parents=True, exist_ok=True)
PREPARED = {}

def discover_dataset(key):
    target = processed / DATASETS[key]
    if target.is_dir() and all((target / f).is_file() for f in REQUIRED_DATA):
        return target.resolve()
    matches = set()
    for base in SEARCH_ROOTS:
        if not base.is_dir():
            continue
        for current, dirs, files in os.walk(base, followlinks=False):
            if not set(REQUIRED_DATA).issubset(files):
                continue
            path = Path(current)
            # Check its own and ancestor names; require exact split/feature filenames.
            label = "/".join(path.parts).lower()
            if key in label or (key == "sports" and "sport" in label):
                matches.add(path.resolve())
    if len(matches) > 1:
        raise RuntimeError(f"Ambiguous {key} sources: {sorted(map(str, matches))}. Narrow SEARCH_ROOTS.")
    return next(iter(matches), None)

for key in DATASETS:
    source = discover_dataset(key)
    target = processed / DATASETS[key]
    if source is None:
        print(f"MISSING {key}: attach its processed data before training it.")
        continue
    link_processed_dataset(source, target)
    PREPARED[key] = source
    print(f"READY {key}: {target} -> {source}")
if not PREPARED:
    raise FileNotFoundError("No complete processed dataset found in SEARCH_ROOTS.")


## 3. Runtime configs and mandatory pre-flight

YAML gốc 256D được copy vào session để ghi rõ horizon, seed và native `CHECKPOINT_FREQ=5` (smoke: 1). Batch, learning rate, gamma, decay và graph không được tự đổi theo GPU. Pre-flight phải chạy thành công toàn bộ tests và **không có skip** trong môi trường Kaggle. Mọi config được compile bằng parser thật để kiểm tra checkpoint frequency và contract 256D.


In [ ]:
import yaml
from stair5_v8_256d_config import validate_capacity_config, tensor_payload_ledger
CONFIG_DIR = SESSION / "configs"
CONFIG_DIR.mkdir()
RUNTIME_CONFIGS = {}
for key, filename in CONFIGS.items():
    original = REPO / "configs" / filename
    values = yaml.safe_load(original.read_text(encoding="utf-8"))
    validate_capacity_config(values)
    values.update(epochs=EPOCHS, eval_freq=EVAL_FREQ, CHECKPOINT_FREQ=5 if MODE == "full" else 1)
    effective = CONFIG_DIR / filename
    effective.write_text(yaml.safe_dump(values, sort_keys=False), encoding="utf-8")
    RUNTIME_CONFIGS[key] = effective
    print(key, "d=", values["embedding_dim"], "batch=", values["batch_size"],
          "CL chunk=", values["cl_chunk_size"], "epochs=", values["epochs"])

preflight = """
import sys, unittest
import models.freerec_compat
suite = unittest.defaultTestLoader.discover("tests", pattern="test_stair5_v8_256d_*.py")
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful() or result.skipped or result.testsRun < 25:
    raise SystemExit("Pre-flight failed or skipped tests; training is blocked.")
from main_stair5_v8_256d import build_config
for path in sys.argv[1:]:
    cfg = build_config(["--config", path])
    assert cfg.embedding_dim == 256 and cfg.epochs > 0
    assert cfg.CHECKPOINT_FREQ == (5 if cfg.epochs == 500 else 1), cfg.CHECKPOINT_FREQ
    assert cfg.eval_freq == (5 if cfg.epochs == 500 else 1), cfg.eval_freq
print("PASS: actual CLI configs and checkpoint frequency")
"""
check = subprocess.run([sys.executable, "-c", preflight, *map(str, RUNTIME_CONFIGS.values())],
                       cwd=REPO, env=SUB_ENV, text=True, capture_output=True)
print(check.stdout)
print(check.stderr)
(SESSION / "preflight.txt").write_text(check.stdout + check.stderr, encoding="utf-8")
check.check_returncode()
PREFLIGHT_OK = True
for key, nu, ni in (("baby", 19445, 7050), ("sports", 35598, 18357),
                   ("electronics", 192403, 63001)):
    ledger = tensor_payload_ledger(nu, ni)
    print(key, "FP32 embeddings + gradients + two Adam moments:",
          round(ledger["embedding_weights_grad_two_adam_moments_bytes_fp32"] / 2**20, 2),
          "MiB (tensor payload only, NOT measured peak)")


## 4. Subprocess runner, checkpoint safety and memory monitoring

Một child process duy nhất mỗi lần chạy. Interrupt sẽ dừng child và monitor; artifact không bị xóa. Checkpoint định kỳ lưu model/optimizer/RNG/coach state. Resume từ `training_checkpoint.pt`, cùng source SHA, data/graph/config và horizon; không resume checkpoint 64D hoặc smoke sang full.

NVML ghi **device-wide used memory** và nếu API hỗ trợ thì cả **child-process used memory** trong toàn bộ preprocessing/train/eval. Coach telemetry ghi **PyTorch training allocated/reserved peaks**. Không dùng memory của kernel notebook làm fallback cho child. NVML không có dữ liệu sẽ ghi unavailable.

Electronics 256D có thể vượt thời gian một session T4. Sau smoke, kiểm tra throughput, peak và checkpoint; dùng archive để tiếp tục cùng run ở session mới.


In [ ]:
import shutil, threading, time
from stair5_v8_256d_notebook_utils import training_command, read_telemetry, read_evaluation, METRICS
RESULTS = {}
ACTIVE_RUN = None

def monitor_memory(pid, target, stopped, interval=2.0):
    nvml = None
    try:
        import pynvml as nvml
        nvml.nvmlInit()
        handle = nvml.nvmlDeviceGetHandleByIndex(NVML_DEVICE_INDEX)
        with Path(target).open("w", encoding="utf-8") as output:
            while not stopped.is_set():
                row = {"time_unix": time.time(), "pid": pid}
                try:
                    row["device_used_mib"] = nvml.nvmlDeviceGetMemoryInfo(handle).used / 2**20
                    try:
                        processes = nvml.nvmlDeviceGetComputeRunningProcesses(handle)
                        used = [p.usedGpuMemory for p in processes if p.pid == pid]
                        if used and used[0] is not None and 0 <= used[0] < 2**63:
                            row["process_used_mib"] = used[0] / 2**20
                    except nvml.NVMLError:
                        pass
                except nvml.NVMLError as error:
                    row["error"] = str(error)
                output.write(json.dumps(row) + "\n")
                output.flush()
                stopped.wait(interval)
    except Exception as error:
        Path(target).write_text(json.dumps({"unavailable": str(error)}) + "\n", encoding="utf-8")
    finally:
        if nvml is not None:
            try:
                nvml.nvmlShutdown()
            except Exception:
                pass

def train_dataset(key, seed=1):
    global ACTIVE_RUN
    if not globals().get("PREFLIGHT_OK"):
        raise RuntimeError("Run pre-flight successfully first.")
    if ACTIVE_RUN is not None:
        raise RuntimeError("Another subprocess is active.")
    if key not in PREPARED:
        raise FileNotFoundError(f"Dataset {key} is not ready; see dataset discovery cell.")
    artifact = SESSION / f"{key}_{MODE}_seed{seed}"
    resume = RESUME_FROM.get((key, seed), "")
    command = training_command(REPO, key, RUNTIME_CONFIGS[key], DATA_ROOT, artifact,
                               OUTPUT_ROOT / "graph_cache", seed=seed, epochs=EPOCHS,
                               eval_freq=EVAL_FREQ, device=DEVICE, resume_from=resume,
                               support_files=SUPPORT_FILES.get(key, ""))
    # Rerunning a finished/failed cell never overwrites evidence.
    artifact.mkdir(exist_ok=False)
    log = artifact / "training.log"
    if resume:
        prior = Path(resume).parent
        # Preserve full curves where available; the inherited Coach trims telemetry to checkpoint epoch.
        for filename in ("training_telemetry.jsonl", "training.log"):
            previous = prior / filename
            if previous.is_file():
                shutil.copy2(previous, artifact / filename)
        if (prior / "manifest.json").is_file():
            shutil.copy2(prior / "manifest.json", artifact / "prior_manifest.json")
    (artifact / "command.json").write_text(json.dumps(command, indent=2), encoding="utf-8")
    print("Running:", subprocess.list2cmdline(command))
    stop = threading.Event()
    started = time.perf_counter()
    process, monitor = None, None
    status = {"dataset": key, "seed": seed, "mode": MODE, "source": SOURCE_SHA,
              "artifact": str(artifact), "log": str(log), "status": "starting"}
    RESULTS[(key, seed)] = status
    try:
        env = SUB_ENV.copy()
        with log.open("a" if resume else "w", encoding="utf-8") as output:
            process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                       errors="replace", bufsize=1)
            ACTIVE_RUN = process
            monitor = threading.Thread(target=monitor_memory,
                args=(process.pid, artifact / "nvml_memory.jsonl", stop), daemon=True)
            monitor.start()
            for line in process.stdout:
                output.write(line)
                output.flush()
                print(line, end="", flush=True)
            returncode = process.wait()
        status.update(exit_code=returncode, status="completed" if returncode == 0 else "failed")
        if returncode != 0:
            raise RuntimeError(f"Training failed ({returncode}); inspect {log}. Artifacts preserved.")
        evaluation = read_evaluation(log)
        if evaluation["best_epoch"] is None or set(evaluation["test_metrics"]) != set(METRICS):
            status["status"] = "metrics_missing"
            raise RuntimeError("Training exited 0 but selected test metrics are missing; inspect log.")
        status.update(best_epoch=evaluation["best_epoch"], test_metrics=evaluation["test_metrics"])
        (artifact / "selected_test_metrics.json").write_text(
            json.dumps(evaluation, indent=2), encoding="utf-8")
        return status
    except BaseException:
        if status["status"] not in ("failed", "metrics_missing"):
            status["status"] = "interrupted_or_failed"
        raise
    finally:
        if process is not None:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            if process.stdout is not None:
                process.stdout.close()
        stop.set()
        if monitor is not None:
            monitor.join(timeout=5)
        ACTIVE_RUN = None
        status["elapsed_seconds"] = time.perf_counter() - started
        status["checkpoint_available"] = (artifact / "training_checkpoint.pt").is_file()
        (artifact / "runner_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
        print("\nStatus:", status["status"], "| checkpoint:", status["checkpoint_available"],
              "| wall minutes:", round(status["elapsed_seconds"] / 60, 2))

def run_all():
    for key in ACTIVE_DATASETS:
        for seed in SEEDS:
            train_dataset(key, seed)
    # Individual training cells below are an alternative; do not run both routes.


## 5. Learning curves and measured memory

Các trường loss đúng: `loss`, `bpr_loss`, `nlgcl_loss`, `weighted_nlgcl_loss`. Memory telemetry `peak_allocated_gib` và `peak_reserved_gib` được đổi sang MiB bằng ×1024. Các curve validation là lịch sử, không phải test dùng để chọn model.

Estimated epoch throughput không bao gồm evaluation/preprocessing/checkpoint. Xem runner wall time để đánh giá thời gian tổng; không suy từ thời gian 64D rằng 256D sẽ kịp quota.


In [ ]:
import matplotlib.pyplot as plt

def plot_dataset(key, seed=1):
    status = RESULTS.get((key, seed))
    if status is None:
        raise ValueError(f"No attempt recorded for {key}, seed={seed}")
    artifact = Path(status["artifact"])
    telemetry = read_telemetry(artifact)
    evaluation = read_evaluation(artifact / "training.log")
    figure, axes = plt.subplots(1, 3, figsize=(18, 4))
    for field in ("loss", "bpr_loss", "weighted_nlgcl_loss"):
        rows = [r for r in telemetry if field in r]
        axes[0].plot([r["epoch"] for r in rows], [r[field] for r in rows], label=field)
    for metric in ("Recall@20", "NDCG@20"):
        rows = [r for r in evaluation["validation"] if metric in r]
        axes[1].plot([r["epoch"] for r in rows], [r[metric] for r in rows], label=metric)
    if evaluation["best_epoch"] is not None:
        axes[1].axvline(evaluation["best_epoch"], linestyle=":", color="grey", label="selected epoch")
    for field in ("peak_allocated_gib", "peak_reserved_gib"):
        rows = [r for r in telemetry if field in r]
        axes[2].plot([r["epoch"] for r in rows], [1024 * r[field] for r in rows], label=field)
    axes[0].set_title("Training losses")
    axes[1].set_title("Validation (no test selection)")
    axes[2].set_title("Training PyTorch peaks (MiB)")
    for axis in axes:
        axis.set_xlabel("Epoch")
        if axis.lines:
            axis.legend()
        axis.grid(alpha=0.25)
    figure.suptitle(f"{key} | V8-256D | {MODE} | seed {seed}")
    figure.tight_layout()
    figure.savefig(artifact / "learning_and_training_memory.png", dpi=150)
    plt.show()
    plt.close(figure)

    path = artifact / "nvml_memory.jsonl"
    samples = [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []
    timed = [r for r in samples if "time_unix" in r]
    if timed:
        figure, axis = plt.subplots(figsize=(10, 3))
        for field in ("device_used_mib", "process_used_mib"):
            rows = [r for r in timed if field in r]
            if rows:
                axis.plot([(r["time_unix"] - timed[0]["time_unix"]) / 60 for r in rows],
                          [r[field] for r in rows], label=field)
        axis.set(xlabel="Wall minutes", ylabel="MiB", title="NVML sampled memory: preprocessing/train/eval")
        if axis.lines:
            axis.legend()
        figure.tight_layout()
        figure.savefig(artifact / "nvml_memory.png", dpi=150)
        plt.show()
        plt.close(figure)
    else:
        print("NVML unavailable; no process/device memory measurement.")
    if telemetry:
        last = telemetry[-1]
        print("Last epoch:", last["epoch"], "| seconds:", last["seconds"],
              "| samples/s:", last["samples_per_second"])
    print("Selected TEST (rounded log values):", evaluation["best_epoch"], evaluation["test_metrics"])


## 6. Sports — independent training cell

Bắt đầu bằng smoke nếu chưa profile 256D. Đổi `MODE` ở cell đầu rồi chạy lại setup để mở session/namespace mới. Sau smoke, chọn `MODE="full"` và chạy lại từ đầu; không nối checkpoint smoke thành full.


In [ ]:
for seed in SEEDS:
    train_dataset("sports", seed)
    plot_dataset("sports", seed)


## 7. Baby — independent training cell


In [ ]:
for seed in SEEDS:
    train_dataset("baby", seed)
    plot_dataset("baby", seed)


## 8. Electronics — independent training cell

Chạy sau smoke/profile, khi đã có thời gian và phương án lưu checkpoint. Mặc định 500 epochs, batch 4096, CL chunk 128; không giảm batch/horizon tự động để báo một kết quả “full”. Nếu cần điều chỉnh tài nguyên, tạo thí nghiệm riêng và ghi rõ protocol khác.


In [ ]:
for seed in SEEDS:
    train_dataset("electronics", seed)
    plot_dataset("electronics", seed)


## 9. Results and historical comparison

Chỉ báo selected TEST từ validation-selected checkpoint. Các số V8-64 bên dưới đã được làm tròn trong log seed 1; delta chỉ là mô tả, không phải kiểm định thống kê. Không so sánh smoke hoặc checkpoint chưa hoàn thành với full500. Kết quả đáng tin cậy cần V8-64/256 matched seeds, mean±std và kiểm tra graph/split/feature hashes từ manifest.


In [ ]:
import csv
V8_64_REFERENCE = {
    "baby": {"Recall@10": 0.0675, "Recall@20": 0.1054, "NDCG@10": 0.0365, "NDCG@20": 0.0462},
    "sports": {"Recall@10": 0.0762, "Recall@20": 0.1142, "NDCG@10": 0.0420, "NDCG@20": 0.0518},
    "electronics": {"Recall@10": 0.0460, "Recall@20": 0.0680, "NDCG@10": 0.0260, "NDCG@20": 0.0317},
}
rows = []
for (key, seed), result in RESULTS.items():
    if MODE != "full" or result["status"] != "completed":
        continue
    for metric in METRICS:
        value = result["test_metrics"][metric]
        reference = V8_64_REFERENCE[key][metric]
        row = {"dataset": key, "seed": seed, "metric": metric, "v8_256d": value,
               "v8_64d_historical_seed1": reference,
               "descriptive_relative_percent": 100 * (value / reference - 1),
               "selected_epoch": result["best_epoch"], "wall_seconds": result["elapsed_seconds"]}
        rows.append(row)
        print(row)
if rows:
    with (SESSION / "metrics_summary.csv").open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
else:
    print("No completed full500 selected-test result available.")
from statistics import mean, stdev
for key in DATASETS:
    completed = [r for (k, seed), r in RESULTS.items()
                 if k == key and r["mode"] == "full" and r["status"] == "completed"]
    if len(completed) > 1:
        print(key, "V8-256D mean +/- sample std across", len(completed), "seeds:")
        for metric in METRICS:
            values = [r["test_metrics"][metric] for r in completed]
            print(metric, mean(values), "+/-", stdev(values))
(SESSION / "session_results.json").write_text(
    json.dumps(list(RESULTS.values()), indent=2), encoding="utf-8")


## 10. Export and resume

Chạy cell này sau một tập hoặc ngay sau khi interrupt. Một archive cho **session hiện tại**, không copy artifact sang thư mục phụ hay tạo hai archive lớn cùng lúc. Checkpoints được giữ để resume; cache lớn nằm ngoài archive (có thể rebuild CPU từ cùng source/data/support). Native FreeRec monitor/best files được gom từ manifest để bảo toàn bằng chứng.

Session khác: gắn archive đã tải xuống, giải nén vào thư mục riêng; đặt `SOURCE_REF` đúng SHA, `RESUME_FROM[(dataset,seed)]` đến `training_checkpoint.pt`, giữ MODE/full500 và cùng config/data. Không dùng `best_model.pt` làm training checkpoint. PyTorch checkpoint là pickle: chỉ load artifact tin cậy của chính thí nghiệm.


In [ ]:
import tarfile
if ACTIVE_RUN is not None:
    raise RuntimeError("Stop the active training process before exporting a consistent archive.")
archive = OUTPUT_ROOT / f"{SESSION_ID}.tar.gz"
(SESSION / "session_results.json").write_text(
    json.dumps(list(RESULTS.values()), indent=2), encoding="utf-8")
native_paths = set()
for result in RESULTS.values():
    artifact = Path(result["artifact"])
    for manifest_name in ("manifest.json", "resume_manifest.json"):
        path = artifact / manifest_name
        if path.is_file():
            native = json.loads(path.read_text(encoding="utf-8")).get("freerec_log_path")
            if native:
                native_path = Path(native)
                if not native_path.is_absolute():
                    native_path = REPO / native_path
                if native_path.is_dir():
                    native_paths.add(native_path.resolve())
with tarfile.open(archive, "w:gz", compresslevel=1) as output:
    output.add(SESSION, arcname=SESSION_ID)
    for index, native in enumerate(sorted(native_paths)):
        output.add(native, arcname=f"{SESSION_ID}/native_freerec/{index}_{native.name}")
print("Export:", archive, "| MiB:", round(archive.stat().st_size / 2**20, 2))
from IPython.display import FileLink, display
display(FileLink(str(archive)))
